# Sponsored Search Ranking — Colab Training
**Two-stage sponsored search:** TF-Ranking listwise model + FAISS HNSW retrieval

Runtime: GPU (T4) recommended. CPU works but ~3× slower.

Steps:
1. Clone repo + install dependencies
2. Generate synthetic dataset
3. Train TF-Ranking model (LambdaLoss)
4. Evaluate NDCG@10, MRR
5. Build FAISS index
6. Download model artifacts

In [ ]:
# Cell 1: Clone repo
!git clone https://github.com/saitejasrivilli/sponsored-search-ranking.git
%cd sponsored-search-ranking

In [ ]:
# Cell 2: Install dependencies
!pip install -q tensorflow tensorflow-recommenders tensorflow-ranking
!pip install -q faiss-gpu numpy pandas scikit-learn fastapi uvicorn pyarrow
print('Dependencies installed')

In [ ]:
# Cell 3: Verify GPU
import tensorflow as tf
print('TF version:', tf.__version__)
print('GPU available:', tf.config.list_physical_devices('GPU'))

In [ ]:
# Cell 4: Generate synthetic dataset
import os
os.makedirs('data', exist_ok=True)
os.makedirs('models', exist_ok=True)

from data.synthetic_data import generate_dataset, train_val_test_split

df = generate_dataset(n_queries=10000, ads_per_query=10)
train_df, val_df, test_df = train_val_test_split(df)
df.to_csv('data/synthetic_search_data.csv', index=False)
print(f'Dataset: {len(df):,} rows')

In [ ]:
# Cell 5: Build TF datasets
from model.ranking_model import SponsoredSearchRanker, build_dataset, FEATURE_NAMES

BATCH_SIZE = 512
train_ds = build_dataset(train_df, batch_size=BATCH_SIZE, shuffle=True)
val_ds   = build_dataset(val_df,   batch_size=BATCH_SIZE, shuffle=False)
test_ds  = build_dataset(test_df,  batch_size=BATCH_SIZE, shuffle=False)

print(f'Train batches: {len(train_ds)}')
print(f'Val batches:   {len(val_ds)}')
print(f'Features: {FEATURE_NAMES}')

In [ ]:
# Cell 6: Build and compile model
model = SponsoredSearchRanker(
    hidden_units=(256, 128, 64),
    dropout_rate=0.2,
)

optimizer = tf.keras.optimizers.Adam(
    learning_rate=tf.keras.optimizers.schedules.CosineDecayRestarts(
        initial_learning_rate=1e-3,
        first_decay_steps=len(train_ds) * 5,
    )
)
model.compile(optimizer=optimizer)

# Warm up
dummy = tf.zeros((1, 10, len(FEATURE_NAMES)))
model(dummy)
print(f'Model parameters: {model.count_params():,}')

In [ ]:
# Cell 7: Train
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath='models/ranker_best.weights.h5',
        monitor='val_ndcg@10', mode='max',
        save_best_only=True, save_weights_only=True, verbose=1,
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_ndcg@10', patience=5, mode='max',
        restore_best_weights=True, verbose=1,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=3, verbose=1
    ),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=callbacks,
    verbose=1,
)

In [ ]:
# Cell 8: Evaluate on test set
results = model.evaluate(test_ds, verbose=1)
metric_names = ['loss', 'ndcg@1', 'ndcg@5', 'ndcg@10', 'mrr']

print('\n=== TEST RESULTS ===')
for name, val in zip(metric_names, results):
    print(f'  {name}: {val:.4f}')

best_ndcg = max(history.history.get('val_ndcg@10', [0]))
print(f'\nBest val NDCG@10: {best_ndcg:.4f}')

In [ ]:
# Cell 9: Plot training curves
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history['loss'],         label='train loss')
axes[0].plot(history.history['val_loss'],     label='val loss')
axes[0].set_title('Loss'); axes[0].legend()

axes[1].plot(history.history.get('ndcg@10',    []), label='train NDCG@10')
axes[1].plot(history.history.get('val_ndcg@10',[]), label='val NDCG@10')
axes[1].set_title('NDCG@10'); axes[1].legend()

plt.tight_layout()
plt.savefig('models/training_curves.png', dpi=150)
plt.show()

In [ ]:
# Cell 10: Save weights
model.save_weights('models/ranker_weights')
print('Weights saved to models/ranker_weights')

In [ ]:
# Cell 11: Build FAISS index
import numpy as np
from pipeline.index_builder import AdSearchIndex

# Extract embeddings from test ads
ad_features = tf.constant(test_df[FEATURE_NAMES].values.astype('float32'))
embeddings  = model.tower(ad_features, training=False).numpy()
ad_ids      = test_df['ad_id'].values.tolist()

print(f'Embeddings shape: {embeddings.shape}')

index = AdSearchIndex(dimension=embeddings.shape[1])
index.add(embeddings, ad_ids)
index.save('models/ad_index.bin')

bench = index.benchmark(n_queries=100, k=50)
print(f'FAISS benchmark: {bench}')

In [ ]:
# Cell 12: Download artifacts
import shutil
shutil.make_archive('model_artifacts', 'zip', 'models/')

from google.colab import files
files.download('model_artifacts.zip')
print('Downloaded model_artifacts.zip')